# E-Commerce Sales Analytics & Profit Prediction
DAV mini project notebook. Run from the `notebooks/` folder; it reuses the functions in `src/` so the notebook and the web app always agree.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import utils, data_preprocessing as dp, eda
sns.set_theme(style='whitegrid')

## 1. Load and inspect the raw data

In [ ]:
raw = dp.load_raw_data(utils.RAW_DATA_PATH if utils.RAW_DATA_PATH.exists() else None)
print(raw.shape)
display(raw.head())
display(raw.dtypes.to_frame('dtype'))
print('Missing values:', int(raw.isna().sum().sum()), '| Duplicate rows:', int(raw.duplicated().sum()))
raw.describe()

## 2. Clean and wrangle
Every step reports how many records it touched.

In [ ]:
clean, steps = dp.clean_data(raw)
display(pd.DataFrame(steps))
print('Raw rows:', len(raw), '-> Clean rows:', len(clean))
clean.head()

## 3. Exploratory analysis

In [ ]:
display(eda.describe_numeric(clean))
for line in eda.generate_insights(clean):
    print('-', line)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(clean['Sales_Amount'], bins=40, ax=axes[0, 0]); axes[0, 0].set_title('Sales amount distribution')
sns.histplot(clean['Profit'], bins=40, ax=axes[0, 1]); axes[0, 1].set_title('Profit distribution')
sns.countplot(x='Quantity', data=clean, ax=axes[1, 0]); axes[1, 0].set_title('Quantity distribution')
sns.histplot(clean['Discount'], bins=26, ax=axes[1, 1]); axes[1, 1].set_title('Discount distribution')
plt.tight_layout(); plt.show()

In [ ]:
sns.heatmap(clean[['Quantity', 'Sales_Amount', 'Discount', 'Profit']].corr(), annot=True, fmt='.2f', cmap='Blues', vmin=-1, vmax=1)
plt.title('Correlation heatmap'); plt.show()

## 4. Train both models
Preprocessing lives inside each Pipeline, so it is fitted on the training split only.

In [ ]:
import train_models as tm
from sklearn.model_selection import train_test_split
X = dp.prepare_model_frame(clean); y = clean[utils.TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=utils.TEST_SIZE, random_state=utils.RANDOM_STATE)
rows = []
for name, pipe in tm.build_models().items():
    pipe.fit(X_train, y_train)
    m, _ = tm.evaluate(pipe, X_train, y_train, X_test, y_test)
    rows.append({'Model': name, **m})
results = pd.DataFrame(rows)
results

## 5. Compare
The best model has the lowest test RMSE.

In [ ]:
print('Best model:', results.sort_values('RMSE').iloc[0]['Model'])
results.set_index('Model')[['MAE', 'RMSE']].plot.bar(rot=0, title='Test error (lower is better)'); plt.show()
results.set_index('Model')[['R2']].plot.bar(rot=0, title='R2 (higher is better)'); plt.show()